# Create and run a local RAG pipeline from scratch

## What is RAG?

**RAG** stands for Retrieval-Augmented Generation.

The goal of RAG is to take information and pass it to an LLM so it can generate outputs based on that info.

* **Retrieval** - Find relevant information given a query, e.g. "what are macronutrients?" -> retrieve passages of text related to macronutrients.
* **Augmented** - We want to take the relevant info from retrieval and augment our input (prompt) to an LLM with that information.
* **Generation** -  Take the first two steps and pass them to an LLM for generative outputs.

## Why RAG?

The main goal of RAG is to improve the generation outputs of LLMs.

1. Prevent hallucinations - LLMs are good at generation good looking text, but that does not mean that it is factually correct.
    * RAG can help LLMs generate information based on relevant passages that are factual
2. Work with custom data - Many base LLMs are trained with internet-scale data (means good understanding with language in general), but means the responses can be generic in nature.
    * RAG helps create specific responses based on specific documents

## Overview of the RAG pipeline

1. Open a PDF document.
2. Format the text of the PDF textbook ready for an embedding model.
3. Embed all the chunks of text in the textbook and turn them into numerical representations which can be stored for use later.
4. Build a retrieval that uses vector search to find relevant chunks of text given a query.
5. Create a prompt that incorporates the retrieved pieces of text.
6. Generate an answer to the query based on the passages of the textbook that were retrieved with an LLM.

**Steps 1-3:** Document preprocessing and embedding creation

**Steps 4-6:** Retrieval and generation (search and answer)

## 1. Document/text processing

Items needed:
* PDF document of choice

Steps:
1. Import the PDF document
2. Process text for embedding (e.g. split into chunks of sentences)

### Import PDF document

In [1]:
import os # file path handling
import requests # download things off the internet

# Get PDF document path
pdf_path = "human-nutrition-text.pdf"

# Download pdf
if not os.path.exists(pdf_path):
    print(f"[INFO] File doesn't exist, downloading...")

    # URL of pdf
    url = "https://pressbooks.oer.hawaii.edu/humannutrition2/open/download?type=pdf"

    # local filename
    filename = pdf_path

    # Send a GET request to the URL
    response = requests.get(url)

    # Check if the request was successful
    if response.status_code == 200:
        # Open the file and save it
        with open(filename, "wb") as file:
            file.write(response.content)
        
        print(f"[INFO] The file has been downloaded and saved as {filename}")
    else:
        print(f"[ERR] Failed to download file. Status code: {response.status_code}")
else:
    print(f"[INFO] File {pdf_path} already exists.")

[INFO] File human-nutrition-text.pdf already exists.


### Open PDF

In [2]:
import fitz # PyMuPDF -> used to read pdf files
from tqdm.auto import tqdm # progress bars

def text_formatter(text: str) -> str:
    """Performs minor formatting on text"""
    
    # remove newlines and extra whitespace chars
    cleaned_text = text.replace("\n", " ").strip()

    return cleaned_text

def read_pdf(path: str) -> list[dict]:
    doc = fitz.open(path)

    pages = []

    # Loop through all the pages in the pdf
    for page_num, page in tqdm(enumerate(doc)):
        text = page.get_text() # get text of current page
        text = text_formatter(text) # format text

        # append all important info about the current page
        pages.append({
            "page_number" : page_num - 41,
            "char_count": len(text),
            "word_count": len(text.split(" ")),
            "sentence_count": len(text.split(". ")),
            "token_count": len(text) / 4, # 1 token ~= 4 chars
            "text": text
        })

    return pages

#### Key term

| Term | Description |
| ----- | ----- | 
| **Token** | A sub-word piece of text. For example, "hello, world!" could be split into ["hello", ",", "world", "!"]. A token can be a whole word,<br> part of a word or group of punctuation characters. 1 token ~= 4 characters in English, 100 tokens ~= 75 words.<br> Text gets broken into tokens before being passed to an LLM. |

In [3]:
pages = read_pdf(path=pdf_path)

import random

random.sample(pages, k=3)

0it [00:00, ?it/s]

[{'page_number': 312,
  'char_count': 748,
  'word_count': 125,
  'sentence_count': 5,
  'token_count': 187.0,
  'text': 'Learning Activities  Technology Note: The second edition of the Human  Nutrition Open Educational Resource (OER) textbook  features interactive learning activities.  These activities are  available in the web-based textbook and not available in the  downloadable versions (EPUB, Digital PDF, Print_PDF, or  Open Document).  Learning activities may be used across various mobile  devices, however, for the best user experience it is strongly  recommended that users complete these activities using a  desktop or laptop computer and in Google Chrome.    An interactive or media element has been  excluded from this version of the text. You can  view it online here:  http://pressbooks.oer.hawaii.edu/ humannutrition2/?p=212    312  |  How Lipids Work'},
 {'page_number': 1101,
  'char_count': 1786,
  'word_count': 314,
  'sentence_count': 20,
  'token_count': 446.5,
  'text': 't

In [4]:
import pandas as pd

df = pd.DataFrame(pages) # convert the dict into a dataframe
df.head()

,page_number,char_count,word_count,sentence_count,token_count,text
0,-41,29,4,1,7.25,Human Nutrition: 2020 Edition
1,-40,0,1,1,0.00,
2,-39,320,54,1,80.00,Human Nutrition: 2020 Edition UNIVERSITY OF ...
3,-38,212,32,1,53.00,Human Nutrition: 2020 Edition by University of...
4,-37,797,147,3,199.25,Contents Preface University of Hawai‘i at Mā...


In [5]:
df.describe().round(2) # generate a statistical summary of the data

,page_number,char_count,word_count,sentence_count,token_count
count,1208.00,1208.00,1208.00,1208.00,1208.00
mean,562.50,1148.00,199.50,10.52,287.00
std,348.86,560.38,95.83,6.55,140.10
min,-41.00,0.00,1.00,1.00,0.00
25%,260.75,762.00,134.00,5.00,190.50
50%,562.50,1231.50,216.00,10.00,307.88
75%,864.25,1603.50,272.00,15.00,400.88
max,1166.00,2308.00,430.00,39.00,577.00


#### Why does the token count matter?

Token count is important to think about because:
1. Embedding models don't deal with infinite tokens.
2. LLMs don't deal with infinite tokens.

The embedding model we will use (`all-mpnet-base-v2`) is trained to embed sequences of 384 tokens.

As for LLMs, they can't accept infinite tokens in their **context window**.

#### Key term

| Term | Description |
| ----- | ----- | 
| **LLM context window** | The number of tokens a LLM can accept as input. For example, as of March 2024, GPT-4 has a default context window of 32k tokens<br> (about 96 pages of text) but can go up to 128k if needed. A recent open-source LLM from Google, Gemma (March 2024) has a context<br> window of 8,192 tokens (about 24 pages of text). A higher context window means an LLM can accept more relevant information<br> to assist with a query. For example, in a RAG pipeline, if a model has a larger context window, it can accept more reference items<br> from the retrieval system to aid with its generation. |

### Preprocess text for chunking

Split pages into sentences by either:
1. Splitting on `". "`
2. Use NLP library (e.g. `spaCy` and `nltk`)

In [6]:
from spacy.lang.en import English

nlp = English() # create a blank English NLP model

# Add a sentencizer pipeline -> turns text into sentences
nlp.add_pipe("sentencizer")

# Create a document instance as an example
doc = nlp("This is the first sentence. This is the second sentence. I like cats.")
assert len(list(doc.sents)) == 3

# Print out the sentences split
list(doc.sents)

[This is the first sentence., This is the second sentence., I like cats.]

In [7]:
pages[67]

{'page_number': 26,
 'char_count': 1799,
 'word_count': 321,
 'sentence_count': 21,
 'token_count': 449.75,
 'text': 'Factors that Drive Food Choices  Along with these influences, a number of other factors affect the  dietary choices individuals make, including:  • Taste, texture, and appearance. Individuals have a wide range  of tastes which influence their food choices, leading some to  dislike milk and others to hate raw vegetables. Some foods that  are very healthy, such as tofu, may be unappealing at first to  many people. However, creative cooks can adapt healthy foods  to meet most people’s taste.  • Economics. Access to fresh fruits and vegetables may be scant,  particularly for those who live in economically disadvantaged  or remote areas, where cheaper food options are limited to  convenience stores and fast food.  • Early food experiences. People who were not exposed to  different foods as children, or who were forced to swallow  every last bite of overcooked vegetables, may

In [8]:
for item in tqdm(pages):
    # Apply the pipeline to the text and get the sentences
    item["sentences"] = list(nlp(item["text"]).sents)

    # Convert all sentences to strings (default is a spaCy datatype)
    item["sentences"] = [str(sentence) for sentence in item["sentences"]]

    # Count the sentences
    item["sentence_count_spacy"] = len(item["sentences"])

  0%|          | 0/1208 [00:00<?, ?it/s]

In [9]:
random.sample(pages, k=1)

[{'page_number': 522,
  'char_count': 1957,
  'word_count': 351,
  'sentence_count': 14,
  'token_count': 489.25,
  'text': 'are packaged into the lipid-containing chylomicrons inside small  intestine mucosal cells and then transported to the liver. In the liver,  carotenoids are repackaged into lipoproteins, which transport them  to cells.  The retinoids are aptly named as their most notable function is  in the retina of the eye where they aid in vision, particularly in  seeing under low-light conditions. This is why night blindness is the  most definitive sign of vitamin A deficiency.Vitamin A has several  important functions in the body, including maintaining vision and a  healthy immune system. Many of vitamin A’s functions in the body  are similar to the functions of hormones (for example, vitamin A can  interact with DNA, causing a change in protein function). Vitamin A  assists in maintaining healthy skin and the linings and coverings of  tissues; it also regulates growth and de

In [10]:
df = pd.DataFrame(pages)
df.describe().round(2)

,page_number,char_count,word_count,sentence_count,token_count,sentence_count_spacy
count,1208.00,1208.00,1208.00,1208.00,1208.00,1208.00
mean,562.50,1148.00,199.50,10.52,287.00,10.32
std,348.86,560.38,95.83,6.55,140.10,6.30
min,-41.00,0.00,1.00,1.00,0.00,0.00
25%,260.75,762.00,134.00,5.00,190.50,5.00
50%,562.50,1231.50,216.00,10.00,307.88,10.00
75%,864.25,1603.50,272.00,15.00,400.88,15.00
max,1166.00,2308.00,430.00,39.00,577.00,28.00


### Chunking

The concept of splitting larger pieces of text into smaller ones is often referred to as text splitting or chunking.

We do this because:
1. Texts are easier to filter (smaller groups are easier to inspect)
2. So text chunks can fit into the embedding model context window (e.g. 384 token limit)
3. The contexts passed to the LLM can be more specific and focused

We will split it into groups of 10 sentences (could also try 5, 7, 8, etc.)

Libraries such as `LangChain` could help with this.

In [11]:
# Define split size to turn groups of sentences into chunks
chunk_size = 10

# Create a function that splits lists of texts recursively into chunk size
# e.g. [20] -> [10, 10]
# e.g. [25] -> [10, 10, 5] (basically do chunk_size until we run out)
def split_list(text: list[str], slice_size: int=chunk_size) -> list[list[str]]:
    return [text[i:i+slice_size] for i in range(0, len(text), slice_size)]

# Test
test_list = list(range(25))
split_list(test_list)

[[0, 1, 2, 3, 4, 5, 6, 7, 8, 9],
 [10, 11, 12, 13, 14, 15, 16, 17, 18, 19],
 [20, 21, 22, 23, 24]]

In [12]:
# Loop through pages and split the sentences into chunks
for item in tqdm(pages):
    # Use the prev. defined function to split the sentences into chunks of 10
    item["sentence_chunks"] = split_list(item["sentences"],
                                         slice_size=chunk_size)
    
    item["num_chunks"] = len(item["sentence_chunks"]) # how many chunks there are

  0%|          | 0/1208 [00:00<?, ?it/s]

In [13]:
random.sample(pages, k=1)

[{'page_number': 526,
  'char_count': 1383,
  'word_count': 235,
  'sentence_count': 14,
  'token_count': 345.75,
  'text': 'higher incidence of lung cancer midway through the study, which  was consequently stopped.2  Vitamin A Toxicity  Vitamin A toxicity, or hypervitaminosis A, is rare. Typically it  requires you to ingest ten times the RDA of preformed vitamin A in  the form of supplements (it would be hard to consume such high  levels from a regular diet) for a substantial amount of time, although  some people may be more susceptible to vitamin A toxicity at lower  doses. The signs and symptoms of vitamin A toxicity include dry,  itchy skin, loss of appetite, swelling of the brain, and joint pain. In  severe cases, vitamin A toxicity may cause liver damage and coma.  Vitamin A is essential during pregnancy, but doses above 3,000  micrograms per day (10,000 international units) have been linked  to an increased incidence of birth defects. Pregnant women should  check the amount of v

In [14]:
df = pd.DataFrame(pages)
df.describe().round(2)

,page_number,char_count,word_count,sentence_count,token_count,sentence_count_spacy,num_chunks
count,1208.00,1208.00,1208.00,1208.00,1208.00,1208.00,1208.00
mean,562.50,1148.00,199.50,10.52,287.00,10.32,1.53
std,348.86,560.38,95.83,6.55,140.10,6.30,0.64
min,-41.00,0.00,1.00,1.00,0.00,0.00,0.00
25%,260.75,762.00,134.00,5.00,190.50,5.00,1.00
50%,562.50,1231.50,216.00,10.00,307.88,10.00,1.00
75%,864.25,1603.50,272.00,15.00,400.88,15.00,2.00
max,1166.00,2308.00,430.00,39.00,577.00,28.00,3.00


### Split each chunk into its own dictionary item

We want to embed each chunk of sentences into its own numerical representation.

Meaning, we can dive specifically into the text sample that is relevant to a query.

In [15]:
import re # regex

# Split each chunk into its own item
chunks = []

# Loop through all the pages
for item in tqdm(pages):
    # Loop through all the chunks for the current page
    for sentence_chunk in item["sentence_chunks"]:
        chunk_dict = {}

        # Get current chunk's page number
        chunk_dict["page_number"] = item["page_number"]

        # Join the lists of sentences back into together into a single paragraph
        joined_sentences = "".join(sentence_chunk).replace("  ", " ").strip() # remove extra spaces
        joined_sentences = re.sub(r'\.([A-Z])', r'. \1', joined_sentences) # ".A" -> ". A"
        
        # Add the paragraph into the dict
        chunk_dict["sentence_chunk"] = joined_sentences

        # Add some stats of the chunks
        chunk_dict["chunk_char_count"] = len(joined_sentences)
        chunk_dict["chunk_word_count"] = len([word for word in joined_sentences.split(" ")])
        chunk_dict["chunk_token_count"] = len(joined_sentences) / 4 # 1 token = 4 chars

        chunks.append(chunk_dict)

len(chunks) # how many chunks there are in total



  0%|          | 0/1208 [00:00<?, ?it/s]

1843

In [16]:
random.sample(chunks, k=1)

[{'page_number': 165,
  'sentence_chunk': 'Percentage Food Item 90–99 Nonfat milk, cantaloupe, strawberries, watermelon, lettuce, cabbage, celery, spinach, squash 80–89 Fruit juice, yogurt, apples, grapes, oranges, carrots, broccoli, pears, pineapple 70–79 Bananas, avocados, cottage cheese, ricotta cheese, baked potato, shrimp 60–69 Pasta, legumes, salmon, chicken breast 50–59 Ground beef, hot dogs, steak, feta cheese 40–49 Pizza 30–39 Cheddar cheese, bagels, bread 20–29 Pepperoni, cake, biscuits 10–19 Butter, margarine, raisins 1–9 Walnuts, dry-roasted peanuts, crackers, cereals, pretzels, peanut butter 0 Oils, sugars Source: National Nutrient Database for Standard Reference, Release 23. US Department of Agriculture, Agricultural Research Service. http://www.ars.usda.gov/ba/bhnrc/ndl. Updated 2010. Accessed September 2017. There is some debate over the amount of water required to maintain health because there is no consistent scientific evidence proving that drinking a particular amou

In [17]:
df = pd.DataFrame(chunks)
df.describe().round(2)

,page_number,chunk_char_count,chunk_word_count,chunk_token_count
count,1843.00,1843.00,1843.00,1843.00
mean,583.38,734.10,112.74,183.52
std,347.79,447.51,71.24,111.88
min,-41.00,12.00,3.00,3.00
25%,280.50,315.00,45.00,78.75
50%,586.00,745.00,115.00,186.25
75%,890.00,1118.00,173.00,279.50
max,1166.00,1830.00,297.00,457.50


### Filter out short chunks

These chunks may be too short to be useful for embedding and retrieval. We can filter them out by checking the number of tokens in each chunk.

In [18]:
# Show random chunks with under 30 tokens (mostly contain headers or links)

min_token_len = 30

# Create a filtered df that contains only the short chunks and loop through the rows
for _, r in df[df["chunk_token_count"] <= min_token_len].sample(5).iterrows():
    # Display the current rows token count and its text
    print(f"Chunk token count: {r['chunk_token_count']} | Text: {r['sentence_chunk']}")


Chunk token count: 16.0 | Text: PART II CHAPTER 2. THE HUMAN BODY Chapter 2. The Human Body | 53
Chunk token count: 27.75 | Text: In exchange, for the reabsorption of sodium and water, potassium is excreted. Regulation of Water Balance | 169
Chunk token count: 9.75 | Text: 920 | Older Adulthood: The Golden Years
Chunk token count: 23.0 | Text: view it online here: http://pressbooks.oer.hawaii.edu/ humannutrition2/?p=301 The Atom | 471
Chunk token count: 20.5 | Text: PART XVI CHAPTER 16. PERFORMANCE NUTRITION Chapter 16. Performance Nutrition | 931


In [19]:
# Filter the dataframe for rows with under 30 tokens

# convert the custom df with chunks passing the min token limit into a dictionary
chunks_over_min_token_len = df[df["chunk_token_count"] > min_token_len].to_dict(orient="records")

random.sample(chunks_over_min_token_len, k=1)

[{'page_number': 135,
  'sentence_chunk': 'Caucasians face greater health risks for the same BMI than African Americans. Calculating BMI To calculate your BMI, multiply your weight in pounds by 703 (conversion factor for converting to metric units) and then divide the product by your height in inches, squared. BMI = [weight (lb) x 703] ÷ height (in)2 or BMI = [weight (kg)] ÷ height (m)2 More Ways to Calculate The National Heart, Lung, and Blood Institute and the CDC have automatic BMI calculators on their websites: • https://www.nhlbisupport.com/bmi/ • https://www.cdc.gov/healthyweight/assessing/bmi/ adult_bmi/english_bmi_calculator/bmi_calculator.html To see how your BMI indicates the weight category you are in, see Table 2.3 “BMI Categories”. Table 2.3 BMI Categories Categories BMI Underweight < 18.5 Healthy or Normal weight 18.5–24.9 Overweight 25–29.9 Obese > 30.0 Indicators of Health: Body Mass Index, Body Fat Content, and Fat Distribution | 135',
  'chunk_char_count': 921,
  'chu

## 2. Embedding creation

Items needed:
* Embedding moodel of choice

Steps:
1. Embed text chunks with embedding model
2. Save embeddings to a file for later retrieval

**Embeddings** are a broad but powerful concept. While humans understand text, machines understand numbers.

Goal:
- Turn the text chunks into numbers, specifically embeddings (useful numerical representation)

The best part about embeddings is that they are a learned representation.

### Import embedding model

In [20]:
from  sentence_transformers import SentenceTransformer # import the embedding model from huggingface

# Model that will be used
model_name = "all-mpnet-base-v2"

model = SentenceTransformer(model_name_or_path=model_name,
                            device="cpu")

# Create a list of sentences to test the model
sentences = ["The Sentence Transformer library provides an easy way to create embeddings",
             "Sentences can be embedded one by one or in a list",
             "I like cats"]

# Encode/embed the sentences using model.encode()
embeddings = model.encode(sentences)

embeddings_dict = dict(zip(sentences, embeddings))

# Print the embeddings
for sentence, embedding in embeddings_dict.items():
    print(f"Sentence: {sentence}")
    print(f"Embedding: {embedding} \n")

C:\Users\ranic\AppData\Roaming\Python\Python312\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
C:\Users\ranic\AppData\Roaming\Python\Python312\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Sentence: The Sentence Transformer library provides an easy way to create embeddings
Embedding: [-3.17512713e-02  3.37267332e-02 -2.52437890e-02  5.22288345e-02
 -2.35249549e-02 -6.19114423e-03  1.35025214e-02 -6.25500977e-02
  7.50832073e-03 -2.29684841e-02  2.98146065e-02  4.57555354e-02
 -3.26700285e-02  1.39847193e-02  4.18013632e-02 -5.92969768e-02
  4.26310301e-02  5.04657347e-03 -2.44552437e-02  3.98591254e-03
  3.55898067e-02  2.78742649e-02  1.84098873e-02  3.67699340e-02
 -2.29961537e-02 -3.01796701e-02  5.99592924e-04 -3.64504047e-02
  5.69104850e-02 -7.49938935e-03 -3.70004512e-02 -3.04365251e-03
  4.64354679e-02  2.36154278e-03  9.06849607e-07  7.00036483e-03
 -3.92289385e-02 -5.95699111e-03  1.38653489e-02  1.87108992e-03
  5.34202717e-02 -6.18613325e-02  2.19613481e-02  4.86051440e-02
 -4.25697304e-02 -1.69858746e-02  5.04178256e-02  1.54733444e-02
  8.12860206e-02  5.07106483e-02 -2.27496885e-02 -4.35720906e-02
 -2.18387460e-03 -2.14091651e-02 -2.01758165e-02  3.0683226

In [21]:
# How many numbers each sentence is being represented with
embeddings[0].shape

(768,)

### Embed each chunk

#### Test out CPU

In [22]:
# %%time

# model.to("cpu")

# for item in tqdm(chunks_over_min_token_len):
#     item["embedding"] = model.encode(item["sentence_chunk"])

#### Test out GPU

In [23]:
%%time

model.to("cuda")

for item in tqdm(chunks_over_min_token_len):
    item["embedding"] = model.encode(item["sentence_chunk"])

  0%|          | 0/1680 [00:00<?, ?it/s]

CPU times: total: 2min 38s
Wall time: 27.8 s


#### Test out GPU with batches

In [28]:
# Create a list of text chunks
text_chunks = [item["sentence_chunk"] for item in chunks_over_min_token_len]

random.sample(text_chunks, k=1)

['An iron supplement is also recommended at this time. Rice is no longer recommended for a first infant food because of its high arsenic content. When cereals are introduced, parents can try baby oats or baby wheat. A guide to infant feeding can be found in the 2019 USDA Infant Nutrition and Feeding Guide at: https://wicworks.fns.usda.gov/ resources/infant-nutrition-and-feeding-guide. Everyday Connection Different cultures have specific food customs. In ancient 838 | Infancy']

In [25]:
len(text_chunks)

1680

In [27]:
%%time

# Embed all texts in batches
text_chunks_embeddings = model.encode(text_chunks, batch_size=16, convert_to_tensor=True)

text_chunks_embeddings

CPU times: total: 1min 18s
Wall time: 13.9 s


tensor([[ 0.0674,  0.0902, -0.0051,  ..., -0.0221, -0.0232,  0.0126],
        [ 0.0552,  0.0592, -0.0166,  ..., -0.0120, -0.0103,  0.0227],
        [ 0.0280,  0.0340, -0.0206,  ..., -0.0054,  0.0213,  0.0313],
        ...,
        [ 0.0771,  0.0098, -0.0122,  ..., -0.0409, -0.0752, -0.0241],
        [ 0.1030, -0.0165,  0.0083,  ..., -0.0574, -0.0283, -0.0295],
        [ 0.0864, -0.0125, -0.0113,  ..., -0.0522, -0.0337, -0.0299]],
       device='cuda:0')

### Save embeddings to file

If the embedding database is very large (e.g. 100k+ chunks), it may be better to use a vector database (e.g. `Weaviate`, `Pinecone`, `Chroma`, etc.) instead of saving to a file.

In [ ]:
# Convert the chunks dict (which contains the embedding for each chunk) into a dataframe to save
text_chunks_embeddings_df = pd.DataFrame(chunks_over_min_token_len)

embeddings_df_path = "text_chunks_embeddings_df.csv" # save path

# Convert the df into a csv and save it
text_chunks_embeddings_df.to_csv(embeddings_df_path, index=False)

### Import and view saved embeddings

In [ ]:
# Import the csv of the embeddings
text_chunks_embeddings_df_loaded = pd.read_csv(embeddings_df_path)
text_chunks_embeddings_df_loaded.head() # print first 5 rows

,page_number,sentence_chunk,chunk_char_count,chunk_word_count,chunk_token_count,embedding
0,-39,Human Nutrition: 2020 Edition UNIVERSITY OF HA...,308,42,77.00,[ 6.74242899e-02 9.02281404e-02 -5.09547861e-...
1,-38,Human Nutrition: 2020 Edition by University of...,210,30,52.50,[ 5.52156419e-02 5.92139252e-02 -1.66167393e-...
2,-37,Contents Preface University of Hawai‘i at Māno...,766,116,191.50,[ 2.79801823e-02 3.39813717e-02 -2.06426494e-...
3,-36,Lifestyles and Nutrition University of Hawai‘i...,941,144,235.25,[ 6.82566985e-02 3.81274670e-02 -8.46854597e-...
4,-35,The Cardiovascular System University of Hawai‘...,998,152,249.50,[ 3.30264494e-02 -8.49768054e-03 9.57159698e-...
